In [0]:
import io
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

import mlflow

ROOT = Path("/Volumes/workspace/default/glass_sample")

SAVE = ROOT / "models" / "rgbd_glass"
CACHE = Path("/tmp/rgbd_gsd_cache_224")

SIZE = 224
EPOCHS = 5
BATCH_SIZE = 8
LEARNING_RATE = 0.001
SEED = 42
PROGRESS_EVERY = 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

assert torch.cuda.is_available(), (
    "GPU NOT DETECTED. Select Serverless GPU"
    "and the AI environment in Databricks"
)

DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)

SAVE.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

train_parquets = sorted(ROOT.glob("train-*.parquet"))
test_parquets = sorted(ROOT.glob("test-*.parquet"))

assert len(train_parquets) == 2, "Expected 2 files bro"
assert len(test_parquets) == 1, "Expected 1 file bro"

def format_time(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def load_image(value):
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            return Image.open(
                io.BytesIO(value["bytes"])
            ).copy()

        path = value.get("path")
        if path and Path(path).is_file():
            return Image.open(path).copy()

    if isinstance(value, (bytes, bytearray, memoryview)):
        return Image.open(io.BytesIO(value)).copy()

    raise ValueError(f"unsupported image data: {type(value)}")


def prepare_sample(row, output_path):
    rgb_img = load_image(row["image"]).convert("RGB")
    depth_img = load_image(row["depth"])
    mask_img = load_image(row["mask"]).convert("L")

    assert rgb_img.size == depth_img.size == mask_img.size, (
        f"mismatched image dimensions: {row['image_id']}"
    )

    rgb = np.asarray(
        rgb_img.resize((SIZE, SIZE), Image.Resampling.BILINEAR),
        dtype=np.uint8
    )

    depth = np.asarray(
        depth_img.resize((SIZE, SIZE), Image.Resampling.NEAREST)
    )

    if depth.ndim == 3:
        if (
            depth.shape[-1] < 3
            or not np.array_equal(depth[..., 0], depth[..., 1])
            or not np.array_equal(depth[..., 0], depth[..., 2])
        ):
            raise ValueError(
                f"unexpected format: {row['image_id']}"
            )
        depth = depth[..., 0]

    depth = depth.astype(np.float32)

    valid = np.isfinite(depth) & (depth > 0)

    scale = (
        float(np.percentile(depth[valid], 99))
        if valid.any() else 1.0
    )
    scale = max(scale, 1e-6)

    normalized_depth = np.clip(
        np.where(valid, depth / scale, 0),
        0, 1
    ).astype(np.float16)

    raw_mask = np.asarray(
        mask_img.resize((SIZE, SIZE), Image.Resampling.NEAREST)
    )

    unique = np.unique(raw_mask)

    if not np.isin(unique, [0, 1, 255]).all():
        raise ValueError(
            f"Unexpected mask labels {unique[:15]} "
            f"for image {row['image_id']}"
        )

    mask = (raw_mask > 0).astype(np.uint8)

    np.savez_compressed(
        output_path,
        rgb=rgb,
        depth=normalized_depth,
        valid=valid.astype(np.uint8),
        mask=mask
    )

def prepare_split(parquet_files, split, expected_count):
    output_dir = CACHE / split
    output_dir.mkdir(parents=True, exist_ok=True)

    paths = []
    seen_ids = set()
    started = time.perf_counter()
    count = 0

    for parquet_path in parquet_files:
        parquet = pq.ParquetFile(str(parquet_path))

        for batch in parquet.iter_batches(
            batch_size=8,
            columns=["image_id", "image", "depth", "mask"]
        ):
            for row in batch.to_pylist():
                image_id = str(row["image_id"])

                if image_id in seen_ids:
                    raise ValueError(
                        f"Duplicate image ID: {image_id}"
                    )

                seen_ids.add(image_id)

                output_path = (
                    output_dir / f"{count:05d}.npz"
                )

                if not output_path.exists():
                    prepare_sample(row, output_path)

                paths.append(output_path)
                count += 1

                if count % 50 == 0 or count == expected_count:
                    elapsed = time.perf_counter() - started
                    eta = (
                        (expected_count - count)
                        * elapsed / max(count, 1)
                    )

                    print(
                        f"PREPROCESS {split}: "
                        f"{count}/{expected_count} | "
                        f"Elapsed {format_time(elapsed)} | "
                        f"ETA {format_time(eta)}",
                        flush=True
                    )

    assert count == expected_count, (
        f"Expected {expected_count} examples, found {count}"
    )

    return paths

print("\npreparing rgbd glass dataset")

all_train = prepare_split(
    train_parquets, "train", 2400
)

all_test = prepare_split(
    test_parquets, "test", 609
)

indices = np.random.default_rng(SEED).permutation(2400)

train_paths = [all_train[i] for i in indices[:2160]]
val_paths = [all_train[i] for i in indices[2160:]]
test_paths = all_test

print("\nDATA READY")
print("Training:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))

class RGBDGlassDataset(Dataset):
    def __init__(self, paths, augment=False):
        self.paths = paths
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with np.load(self.paths[index]) as sample:
            rgb = sample["rgb"].astype(np.float32) / 255
            depth = sample["depth"].astype(np.float32)
            valid = sample["valid"].astype(np.float32)
            mask = sample["mask"].astype(np.float32)

        if self.augment and random.random() < 0.5:
            rgb, depth, valid, mask = [
                np.flip(a, axis=1).copy()
                for a in (rgb, depth, valid, mask)
            ]

        if self.augment:
            rgb = np.clip(
                rgb * random.uniform(0.9, 1.1),
                0, 1
            )
        x = np.concatenate([
            rgb,
            depth[..., None],
            valid[..., None]
        ], axis=-1).astype(np.float32)

        return (
            torch.from_numpy(x.transpose(2, 0, 1).copy()),
            torch.from_numpy(mask[None].copy())
        )


train_ds = RGBDGlassDataset(train_paths, augment=True)
val_ds = RGBDGlassDataset(val_paths)
test_ds = RGBDGlassDataset(test_paths)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

def conv_block(input_channels, output_channels):
    return nn.Sequential(
        nn.Conv2d(
            input_channels, output_channels,
            3, padding=1, bias=False
        ),
        nn.BatchNorm2d(output_channels),
        nn.ReLU(inplace=True),

        nn.Conv2d(
            output_channels, output_channels,
            3, padding=1, bias=False
        ),
        nn.BatchNorm2d(output_channels),
        nn.ReLU(inplace=True)
    )


class RGBDUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = conv_block(5, 16)
        self.enc2 = conv_block(16, 32)
        self.enc3 = conv_block(32, 64)

        self.middle = conv_block(64, 128)

        self.dec3 = conv_block(128 + 64, 64)
        self.dec2 = conv_block(64 + 32, 32)
        self.dec1 = conv_block(32 + 16, 16)

        self.output = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        a = self.enc1(x)
        b = self.enc2(F.avg_pool2d(a, 2))
        c = self.enc3(F.avg_pool2d(b, 2))

        d = self.middle(F.avg_pool2d(c, 2))

        d = F.interpolate(
            d, size=c.shape[-2:],
            mode="bilinear", align_corners=False
        )
        d = self.dec3(torch.cat([d, c], dim=1))

        d = F.interpolate(
            d, size=b.shape[-2:],
            mode="bilinear", align_corners=False
        )
        d = self.dec2(torch.cat([d, b], dim=1))

        d = F.interpolate(
            d, size=a.shape[-2:],
            mode="bilinear", align_corners=False
        )
        d = self.dec1(torch.cat([d, a], dim=1))

        return self.output(d)

def dice_loss(logits, targets):
    probs = torch.sigmoid(logits)

    intersection = (
        probs * targets
    ).sum(dim=(1, 2, 3))

    denominator = (
        probs.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    return (
        1 - (2 * intersection + 1) / (denominator + 1)
    ).mean()


def evaluate(model, loader):
    model.eval()

    tp = 0
    fp = 0
    fn = 0
    total_loss = 0.0
    batches = 0

    with torch.inference_mode():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(x)

            loss = (
                loss_fn(logits.float(), y)
                + dice_loss(logits.float(), y)
            )

            total_loss += loss.item()
            batches += 1

            prediction = torch.sigmoid(logits) >= 0.5
            truth = y >= 0.5

            tp += int((prediction & truth).sum().item())
            fp += int((prediction & ~truth).sum().item())
            fn += int((~prediction & truth).sum().item())

    union = tp + fp + fn

    return {
        "loss": total_loss / max(batches, 1),
        "iou": tp / max(union, 1),
        "dice": 2 * tp / max(2 * tp + fp + fn, 1),
        "recall": tp / max(tp + fn, 1),
        "precision": tp / max(tp + fp, 1)
    }

positive = 0
total = 0

for path in train_paths:
    with np.load(path) as sample:
        positive += int(sample["mask"].sum())
        total += int(sample["mask"].size)

assert positive > 0, "No glass pixels found in training data"

pos_weight = min(
    6.0,
    max(1.0, (total - positive) / positive)
)

print("\nGlass pixel fraction:", round(positive / total, 4))
print("Positive class weight:", round(pos_weight, 2))

model = RGBDUNet().to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_weight, device=DEVICE)
)

grad_scaler = torch.amp.GradScaler("cuda")

best_iou = -1.0
training_started = time.perf_counter()
last_validation_seconds = 0.0

run = None

try:
    run = mlflow.start_run(
        run_name="rgbd_glass_gpu_5epochs",
        nested=mlflow.active_run() is not None
    )

    mlflow.log_params({
        "dataset": "RGBD-GSD",
        "architecture": "RGBDUNet_5channel",
        "epochs": EPOCHS,
        "image_size": SIZE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "training_samples": len(train_ds),
        "validation_samples": len(val_ds),
        "test_samples": len(test_ds),
        "GPU": torch.cuda.get_device_name(0)
    })

except Exception as error:
    print("MLflow warning", error)

try:
    for epoch in range(1, EPOCHS + 1):
        model.train()

        epoch_started = time.perf_counter()
        cumulative_loss = 0.0

        for step, (x, y) in enumerate(train_loader, 1):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(x)

                loss = (
                    loss_fn(logits.float(), y)
                    + dice_loss(logits.float(), y)
                )

            grad_scaler.scale(loss).backward()
            grad_scaler.step(optimizer)
            grad_scaler.update()

            cumulative_loss += loss.item()

            # ETA UPDATE EVERY 10 BATCHES
            if (
                step % PROGRESS_EVERY == 0
                or step == len(train_loader)
            ):
                torch.cuda.synchronize()

                elapsed_epoch = (
                    time.perf_counter() - epoch_started
                )

                avg_batch_time = elapsed_epoch / step

                remaining_batches = (
                    len(train_loader) - step
                    + (EPOCHS - epoch) * len(train_loader)
                )

                estimated_remaining = (
                    remaining_batches * avg_batch_time
                    + (EPOCHS - epoch + 1)
                    * last_validation_seconds
                )

                total_elapsed = (
                    time.perf_counter() - training_started
                )

                print(
                    f"Epoch {epoch}/{EPOCHS} | "
                    f"Batch {step}/{len(train_loader)} | "
                    f"Loss {cumulative_loss / step:.4f} | "
                    f"Elapsed {format_time(total_elapsed)} | "
                    f"ETA {format_time(estimated_remaining)}",
                    flush=True
                )

        print(
            f"\nvalidating epoch {epoch}...",
            flush=True
        )

        validation_started = time.perf_counter()

        metrics = evaluate(model, val_loader)

        torch.cuda.synchronize()

        last_validation_seconds = (
            time.perf_counter() - validation_started
        )

        avg_train_loss = (
            cumulative_loss / len(train_loader)
        )

        print(
            f"\nEPOCH {epoch}/{EPOCHS} COMPLETE\n"
            f"Train loss: {avg_train_loss:.4f}\n"
            f"Validation IoU: {metrics['iou']:.4f}\n"
            f"Validation Dice: {metrics['dice']:.4f}\n"
            f"Validation Recall: {metrics['recall']:.4f}\n"
            f"Validation Precision: {metrics['precision']:.4f}\n",
            flush=True
        )

        if run is not None:
            mlflow.log_metrics({
                "train_loss": avg_train_loss,
                "val_iou": metrics["iou"],
                "val_dice": metrics["dice"],
                "val_recall": metrics["recall"],
                "val_precision": metrics["precision"]
            }, step=epoch)

        if metrics["iou"] > best_iou:
            best_iou = metrics["iou"]

            torch.save({
                "state_dict": model.state_dict(),
                "epoch": epoch,
                "val_iou": best_iou,
                "size": SIZE,
                "input_channels": [
                    "R", "G", "B",
                    "normalized_depth",
                    "depth_validity"
                ]
            }, SAVE / "glass_rgbd_best.pt")

            print(
                "Saved better checkpoint",
                flush=True
            )

    print(
        "\nALL 5 EPOCHS COMPLETED",
        "Total training time:",
        format_time(time.perf_counter() - training_started)
    )

    checkpoint = torch.load(
        SAVE / "glass_rgbd_best.pt",
        map_location="cpu",
        weights_only=True
    )

    model.load_state_dict(checkpoint["state_dict"])
    model.eval()

    print("\nEvaluating.............")

    test_metrics = evaluate(model, test_loader)

    print("FINAL TEST RESULTS:")
    for name, value in test_metrics.items():
        print(name, round(value, 4))

    if run is not None:
        mlflow.log_metrics({
            f"test_{name}": value
            for name, value in test_metrics.items()
        })

    cpu_model = RGBDUNet().cpu().eval()
    cpu_model.load_state_dict(checkpoint["state_dict"])

    with torch.inference_mode():
        traced_model = torch.jit.trace(
            cpu_model,
            torch.zeros(1, 5, SIZE, SIZE)
        )

    traced_model.save(
        str(SAVE / "glass_rgbd_traced.pt")
    )

    config = {
        "model": "RGBDUNet",
        "input_shape_nchw": [1, 5, SIZE, SIZE],
        "channels": [
            "red", "green", "blue",
            "normalized_depth", "depth_validity"
        ],
        "rgb_range": "[0,1]",
        "depth_normalization": (
            "Divide positive depth readings by each "
            "frame's 99th percentile. Clip to [0,1]."
        ),
        "validity": (
            "1 for finite positive depth, 0 otherwise"
        ),
        "output": (
            "Single channel of logits. Apply sigmoid; "
            "threshold 0.5 for binary glass mask."
        ),
        "dataset": "RGBD-GSD",
        "dataset_license": "CC BY-NC 4.0",
        "note": (
            "Requires validation on iPhone LiDAR data "
            "and conversion to Core ML"
        )
    }

    (SAVE / "model_config.json").write_text(
        json.dumps(config, indent=2)
    )

    sample_x, actual_mask = test_ds[0]

    with torch.inference_mode():
        probability = torch.sigmoid(
            model(
                sample_x.unsqueeze(0).to(DEVICE)
            )
        )[0, 0].cpu().numpy()

    predicted_mask = probability >= 0.5

    rgb = sample_x[:3].permute(1, 2, 0).numpy()

    overlay = rgb.copy()
    overlay[predicted_mask] = (
        0.45 * overlay[predicted_mask]
        + 0.55 * np.array([0.65, 0.25, 1.0])
    )

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(rgb)
    axes[0].set_title("RGB image")

    axes[1].imshow(
        actual_mask[0].numpy(),
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[1].set_title("Actual glass mask")

    axes[2].imshow(overlay)
    axes[2].set_title("Predicted glass overlay")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()

    fig.savefig(
        SAVE / "heldout_preview.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

    if run is not None:
        for filename in [
            "glass_rgbd_best.pt",
            "glass_rgbd_traced.pt",
            "model_config.json",
            "heldout_preview.png"
        ]:
            mlflow.log_artifact(str(SAVE / filename))

    print("\nGLASS MODEL TRAINING FINISHED!!!!!!!!!!!")
    print("Saved model files to:", SAVE)

finally:
    if run is not None:
        mlflow.end_run()

In [0]:
from pathlib import Path

ROOT = Path("/Volumes/workspace/default/glass_sample")

print("SEARCHING:", ROOT)
print("=" * 60)

for file in ROOT.rglob("*"):
    if file.is_file():
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"{file}  ({size_mb:.1f} MB)")

print("\n" + "=" * 60)
print("PARQUET FILES FOUND:")

parquet_files = sorted(ROOT.rglob("*.parquet"))

for file in parquet_files:
    print(file)

print(f"\nTotal Parquet files: {len(parquet_files)}")

In [0]:
import os
import importlib.util

print("Current folder:", os.getcwd())
print("MLflow location:", importlib.util.find_spec("mlflow"))

In [0]:
import mlflow
import torch

print("MLflow version:", mlflow.__version__)
print("MLflow location:", mlflow.__file__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [0]:
import traceback
import importlib.metadata

for package in ["mlflow", "mlflow-skinny", "mlflow-tracing"]:
    try:
        print(package, importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        print(package, "not installed")

try:
    import mlflow
    print("MLflow imported successfully")
except Exception:
    traceback.print_exc()

In [0]:
%pip install --upgrade "mlflow[databricks]==3.17.2"

In [0]:
%restart_python

In [0]:
import mlflow
import torch

print("MLflow:", mlflow.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [0]:
%pip install torch torchvision

In [0]:
dbutils.library.restartPython()

In [0]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())